In [ ]:
# Mount Google Drive (if needed for saving model and data)
# from google.colab import drive
# drive.mount('/content/drive')

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import re
import string
import tensorflow_datasets as tfds
from tensorflow.keras.layers import Embedding, LSTM, Dense, GRU
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
import os

2025-03-07 11:26:06.696048: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-07 11:26:17.029408: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-07 11:26:17.029519: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-07 11:26:18.807915: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-07 11:26:24.802272: I tensorflow/core/platform/cpu_feature_guar

ModuleNotFoundError: No module named 'tensorflow_datasets'

In [ ]:
# Load dataset
def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    data = []
    for line in lines:
        parts = line.strip().split('\t')
        if len(parts) >= 2:  # Ensuring valid data
            data.append((parts[0], parts[1]))
    return pd.DataFrame(data, columns=['English', 'Bangla'])# Load dataset
def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    data = []
    for line in lines:
        parts = line.strip().split('\t')
        if len(parts) >= 2:  # Ensuring valid data
            data.append((parts[0], parts[1]))
    return pd.DataFrame(data, columns=['English', 'Bangla'])
data_path = '/media/nsl47/hdd/Robotics_for_kids/Updated/New/For_client/Final/Materials/Others/Play_with_NLP/Project_1/Bangla_English_Translator/data/dataset.txt'  # Update with actual file path
df = load_dataset(data_path)
print("Dataset Sample:")
print(df.head())

Dataset Sample:
  English  Bangla
0     Go.    যাও।
1     Go.    যান।
2     Go.     যা।
3    Run!  পালাও!
4    Run!  পালান!


In [ ]:
# Preprocessing functions
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{string.punctuation}]", "", text)
    return text

df['English'] = df['English'].apply(clean_text)
df['Bangla'] = df['Bangla'].apply(clean_text)

In [ ]:
print(df['English'].head(10))

0      go
1      go
2      go
3     run
4     run
5     who
6     wow
7    fire
8    help
9    help
Name: English, dtype: object


In [ ]:
print(df['Bangla'].head(10))

0      যাও।
1      যান।
2       যা।
3     পালাও
4     পালান
5        কে
6       বাহ
7      আগুন
8    বাঁচাও
9    বাঁচান
Name: Bangla, dtype: object


In [ ]:
len(df['English'])

6509

In [ ]:
len(df['Bangla'])

6509

In [ ]:
# Tokenization
eng_tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    df['English'], target_vocab_size=2**13)
bn_tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    df['Bangla'], target_vocab_size=2**13)


In [ ]:
print(eng_tokenizer)

<SubwordTextEncoder vocab_size=3742>


In [ ]:
print(bn_tokenizer)

<SubwordTextEncoder vocab_size=1464>


In [ ]:
# Convert text to sequences
def encode_texts(eng, bn):
    eng_seq = eng_tokenizer.encode(eng)
    bn_seq = bn_tokenizer.encode(bn)
    return eng_seq, bn_seq

df['eng_seq'], df['bn_seq'] = zip(*df.apply(lambda row: encode_texts(row['English'], row['Bangla']), axis=1))

In [ ]:
df['eng_seq'].head(10)

0     [106]
1     [106]
2     [106]
3     [506]
4     [506]
5    [1913]
6    [1885]
7     [475]
8     [310]
9     [310]
Name: eng_seq, dtype: object

In [ ]:
df['bn_seq'].head(10)

0        [13, 1, 88, 15]
1         [13, 1, 7, 15]
2               [13, 40]
3     [24, 1, 18, 1, 88]
4      [24, 1, 18, 1, 7]
5                 [8, 2]
6            [19, 1, 52]
7           [105, 16, 7]
8    [19, 70, 25, 1, 88]
9     [19, 70, 25, 1, 7]
Name: bn_seq, dtype: object

In [ ]:
# Padding sequences
max_len = max(max(df['eng_seq'].apply(len)), max(df['bn_seq'].apply(len)))
df['eng_seq'] = df['eng_seq'].apply(lambda x: x + [0] * (max_len - len(x)))
df['bn_seq'] = df['bn_seq'].apply(lambda x: x + [0] * (max_len - len(x)))


In [ ]:
# Splitting dataset
X_train, X_test, y_train, y_test = train_test_split(df['eng_seq'].tolist(), df['bn_seq'].tolist(), test_size=0.1)
X_train, X_test, y_train, y_test = map(np.array, [X_train, X_test, y_train, y_test])


In [ ]:

# Build Seq2Seq Model
class Seq2Seq(Model):
    def __init__(self, vocab_size, embedding_dim, units):
        super(Seq2Seq, self).__init__()
        self.encoder = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(units, return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.Dense(vocab_size, activation='softmax')

    def call(self, inputs):
        x = self.encoder(inputs)
        x, state = self.gru(x)
        x = self.decoder(x)
        return x


In [ ]:

# Define hyperparameters
embedding_dim = 256
units = 512
vocab_size = max(eng_tokenizer.vocab_size, bn_tokenizer.vocab_size) + 1


In [ ]:
# Initialize and compile model
model = Seq2Seq(vocab_size, embedding_dim, units)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [ ]:
# Train model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=32)


Epoch 1/100
184/184 [==============================] - 72s 384ms/step - loss: 1.3710 - accuracy: 0.8005 - val_loss: 1.0243 - val_accuracy: 0.8113
Epoch 2/100
184/184 [==============================] - 68s 370ms/step - loss: 0.9613 - accuracy: 0.8156 - val_loss: 0.9429 - val_accuracy: 0.8163
Epoch 3/100
184/184 [==============================] - 58s 313ms/step - loss: 0.9099 - accuracy: 0.8187 - val_loss: 0.9384 - val_accuracy: 0.8175
Epoch 4/100
184/184 [==============================] - 56s 304ms/step - loss: 0.8875 - accuracy: 0.8201 - val_loss: 0.9473 - val_accuracy: 0.8181
Epoch 5/100
184/184 [==============================] - 62s 337ms/step - loss: 0.8690 - accuracy: 0.8217 - val_loss: 0.9106 - val_accuracy: 0.8197
Epoch 6/100
184/184 [==============================] - 55s 302ms/step - loss: 0.8489 - accuracy: 0.8238 - val_loss: 0.9095 - val_accuracy: 0.8202
Epoch 7/100
184/184 [==============================] - 59s 318ms/step - loss: 0.8334 - accuracy: 0.8249 - val_loss: 0.9020 -

In [ ]:
model.save_weights('models/bnen_weights')

In [ ]:
model.save('models/bnen_models', save_format='tf')

INFO:tensorflow:Assets written to: models/bnen_models/assets


INFO:tensorflow:Assets written to: models/bnen_models/assets


In [ ]:
# Function to translate English to Bangla
def translate(sentence):
    sentence = clean_text(sentence)
    seq = eng_tokenizer.encode(sentence)
    seq = seq + [0] * (max_len - len(seq))
    seq = np.array([seq])
    pred = model.predict(seq)
    pred_seq = np.argmax(pred, axis=-1)[0]
    translated_text = bn_tokenizer.decode([i for i in pred_seq if i != 0])
    return translated_text

# Example translation
example = "How are you"
print(f"English: {example}")
print(f"Bangla Translation: {translate(example)}")


English: How are you


NameError: name 'clean_text' is not defined

In [ ]:
# Function to translate English to Bangla
def translate(sentence):
    sentence = clean_text(sentence)
    seq = eng_tokenizer.encode(sentence)
    seq = seq + [0] * (max_len - len(seq))
    seq = np.array([seq])
    pred = model.predict(seq)
    pred_seq = np.argmax(pred, axis=-1)[0]
    translated_text = bn_tokenizer.decode([i for i in pred_seq if i != 0])
    return translated_text

# Example translations
examples = [
    "How are you",
    "Good morning",
    "What is your name",
    "I am learning Python",
    "Where is the nearest hospital",
    "I love programming",
    "Can you help me",
    "How much is this",
    "I am feeling good today",
    "What time is it"
]

# Print English and Bangla translations
for example in examples:
    print(f"English: {example}")
    print(f"Bangla Translation: {translate(example)}")
    print("-" * 50)


English: How are you


NameError: name 'clean_text' is not defined

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu

nltk.download('punkt')  # Ensure necessary resources are downloaded

import re

def clean_text(text):
    """
    Basic text cleaning function.
    - Lowercases text
    - Removes special characters (except spaces and basic punctuation)
    """
    text = text.lower()  # Convert to lowercase
    text = re.sub(r"[^a-zA-Z0-9অ-ঔক-হৎংঃঁ\s.,?!]", "", text)  # Remove unwanted characters
    text = text.strip()  # Remove leading/trailing spaces
    return text


def translate(sentence):
    sentence = clean_text(sentence)
    seq = eng_tokenizer.encode(sentence)
    seq = seq + [0] * (max_len - len(seq))
    seq = np.array([seq])
    pred = model.predict(seq)
    pred_seq = np.argmax(pred, axis=-1)[0]
    translated_text = bn_tokenizer.decode([i for i in pred_seq if i != 0])
    return translated_text

# Example translations
examples = [
    "How are you",
    "Good morning",
    "What is your name",
    "I am learning Python",
    "Where is the nearest hospital",
    "I love programming",
    "Can you help me",
    "How much is this",
    "I am feeling good today",
    "What time is it"
]

# Reference translations (manually provided or from a dataset)
reference_translations = [
    ["আপনি কেমন আছেন"],  # How are you
    ["সুপ্রভাত"],  # Good morning
    ["আপনার নাম কি"],  # What is your name
    ["আমি পাইথন শিখছি"],  # I am learning Python
    ["নিকটতম হাসপাতাল কোথায়"],  # Where is the nearest hospital
    ["আমি প্রোগ্রামিং ভালোবাসি"],  # I love programming
    ["আপনি কি আমাকে সাহায্য করতে পারেন"],  # Can you help me
    ["এর দাম কত"],  # How much is this
    ["আমি আজ ভালো অনুভব করছি"],  # I am feeling good today
    ["এখন কত সময়"],  # What time is it
]

# Compute BLEU score
predictions = []
for example in examples:
    predicted_translation = translate(example)
    predictions.append(predicted_translation.split())  # Tokenize predictions
    print(f"English: {example}")
    print(f"Bangla Translation: {predicted_translation}")
    print("-" * 50)

# Tokenize reference translations correctly
tokenized_references = [[ref[0].split()] for ref in reference_translations]

# Calculate BLEU score for each translation
bleu_scores = [sentence_bleu(ref, pred) for ref, pred in zip(tokenized_references, predictions)]

# Compute average BLEU score
average_bleu = sum(bleu_scores) / len(bleu_scores)

print(f"\nAverage BLEU Score: {average_bleu:.4f}")


1/1 [==============================] - 0s 51ms/step
English: How are you
Bangla Translation: তি ই েমন মন মন আছো
--------------------------------------------------
1/1 [==============================] - 0s 44ms/step


[nltk_data] Downloading package punkt to /home/nsl47/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


English: Good morning
Bangla Translation: সবুপ্রভাত।
--------------------------------------------------
1/1 [==============================] - 0s 48ms/step
English: What is your name
Bangla Translation: আপনার নাম কী
--------------------------------------------------
1/1 [==============================] - 0s 36ms/step
English: I am learning Python
Bangla Translation: আমি ি একটাযি আমাসা একটেখছ।
--------------------------------------------------
1/1 [==============================] - 0s 42ms/step
English: Where is the nearest hospital
Bangla Translation: তাফটটছাতাতাতার মধে তাতাতালটা কোথাযাযে
--------------------------------------------------
1/1 [==============================] - 0s 44ms/step
English: I love programming
Bangla Translation: আমি অসুল্লোলালালোলোসি।ি।
--------------------------------------------------
1/1 [==============================] - 0s 66ms/step
English: Can you help me
Bangla Translation: আপনি কি আমাকে সাহায্য করতে পারবেন
----------------------------------------------

/home/nsl47/anaconda3/envs/yolo_env/lib/python3.8/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/nsl47/anaconda3/envs/yolo_env/lib/python3.8/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/nsl47/anaconda3/envs/yolo_env/lib/python3.8/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consid